# 1.4. CNN gėlių nuotraukų klasifikavimui

**Liutauras Turauskis · DIPf-24 · ND1 tęsinys**

Pritaikyta pagal dėstytojo pateiktą `cnn.ipynb`. Pavyzdžio eiga paaiškinta prie atitinkamų žingsnių;
programos komentarai paaiškina svarbiausias operacijas. Išsaugota CNN ir MLP mokymo eiga, filtrų bei požymių žemėlapių analizė ir rankinis konvoliucijos skaičiavimas.

**Paleidimas:** Google Colab → „Runtime“ → „Run all“. Ląsteles vykdyti nuo pradžios.
Duomenys atsisiunčiami automatiškai; savo kodo, failų ir API rakto įterpti nereikia.
GPU rekomenduojamas vaizdų mokymui, bet nėra būtinas. Pirmas atsisiuntimas gali užtrukti.

**Rezultatų sąžiningumas:** šioje pateikiamoje kopijoje mokymo išvestys išvalytos.
Rodikliai, grafikai ir konkrečios išvados apskaičiuojami paleidus, o ne įrašomi iš anksto.
Parengimo aplinkoje TensorFlow mokymas nebuvo vykdytas; prieš pateikiant reikia paleisti visą užrašyną
Colab ir išsaugoti kopiją su išvestimis. Atsitiktinė sėkla mažina rezultatų svyravimą, tačiau GPU
ir bibliotekų versijos gali lemti nedidelius skirtumus.


In [ ]:
%pip -q install "pandas>=2.2,<3" "scikit-learn>=1.5,<2" "matplotlib>=3.8,<4" requests pillow


In [ ]:
from pathlib import Path
import io, json, zipfile, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report)
from IPython.display import display
SEED = 42
OUT = Path('rezultatai')
OUT.mkdir(exist_ok=True)


In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))


## 1. Pasirinkti vaizdų duomenys

Originalo CIFAR-10 / interneto nuotraukos pakeistos **Flowers Photos** rinkiniu:
`daisy`, `dandelion`, `roses`, `sunflowers`, `tulips` (5 klasės).
[TensorFlow šaltinis](https://www.tensorflow.org/tutorials/load_data/images).
Visas archyvas užima apie 218 MB. Nuotraukų autoriai ir CC BY licencijos pateikti archyvo `LICENSE.txt`.
Tai vieši pasirinkti duomenys, o ne mano paties fotografuotos nuotraukos.

Eksperimentui atsitiktinai pasirenkama iki 200 nuotraukų iš kiekvienos klasės.
Pasikartojantys vienodi failų turiniai atmetami prieš skaidymą. Mokymo, validavimo ir testo dalys:
70 %, 15 %, 15 %. Visuose trijuose vaizdų užrašynuose parinkimo taisyklės vienodos.
Panašių, bet ne identiškų nuotraukų šis patikrinimas neaptinka.


In [ ]:
from PIL import Image, ImageDraw
from collections import Counter
import tarfile
CACHE = Path('/content/nd1_duomenys') if Path('/content').exists() else Path('nd1_duomenys')
CACHE.mkdir(exist_ok=True)
archive = keras.utils.get_file(fname='flower_photos.tgz',
    origin='https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz',
    cache_dir=str(CACHE.resolve()), cache_subdir='archyvai', extract=False)
DATA = CACHE / 'flower_photos'
if not DATA.exists():
    with tarfile.open(archive) as tar:
        tar.extractall(CACHE, filter='data')
classes = sorted(p.name for p in DATA.iterdir() if p.is_dir())
assert classes == ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']
rng = np.random.default_rng(SEED)
rows, seen = [], set()
for label, name in enumerate(classes):
    files = sorted((DATA / name).glob('*.jpg'))
    # Atsitiktinimas atliekamas po rūšiavimo, kad failų sistemos tvarka nekeistų imties.
    files = [files[int(i)] for i in rng.permutation(len(files))]
    selected = 0
    for path in files:
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen:
            continue
        seen.add(digest)
        rows.append({'path': str(path), 'relative_path': str(path.relative_to(DATA)),
                     'label': label, 'class_name': name, 'sha256': digest})
        selected += 1
        if selected == 200:
            break
images = pd.DataFrame(rows)
train_img, remaining = train_test_split(images, test_size=.30, stratify=images.label, random_state=SEED)
val_img, test_img = train_test_split(remaining, test_size=.50, stratify=remaining.label, random_state=SEED)
assert set(train_img.sha256).isdisjoint(val_img.sha256)
assert set(train_img.sha256).isdisjoint(test_img.sha256)
assert set(val_img.sha256).isdisjoint(test_img.sha256)
manifest = pd.concat([part.assign(split=name) for name, part in
    [('train', train_img), ('val', val_img), ('test', test_img)]])
manifest.drop(columns='path').to_csv(OUT / 'geliu_padalinimas.csv', index=False)
display(pd.crosstab(manifest.class_name, manifest.split))
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, name in zip(axes, classes):
    path = train_img.loc[train_img.class_name == name, 'path'].iloc[0]
    ax.imshow(Image.open(path).convert('RGB')); ax.set_title(name); ax.axis('off')
plt.tight_layout(); plt.show()


## 2. Įvestis ir konvoliucinis tinklas

Originale buvo 32×32 CIFAR-10 vaizdai ir 10 klasių, čia 96×96 RGB vaizdai ir 5 klasės.
`Rescaling` reikšmes 0..255 pakeičia į 0..1. `Conv2D` mokosi vietinių raštų, o `MaxPooling2D`
mažina žemėlapių dydį. `Flatten` išskleidžia požymius, tankieji sluoksniai juos paverčia klasėmis.
Paskutinis softmax sluoksnis grąžina penkis tikimybių įverčius; etiketės yra sveikieji 0..4.


In [ ]:
SIZE = 96  # Vienoda įvestis CNN ir MLP palyginimui.
BATCH = 32
def image_dataset(frame, training=False, size=SIZE):
    ds = tf.data.Dataset.from_tensor_slices((frame.path.to_numpy(str), frame.label.to_numpy('int32')))
    def decode(path, label):
        image = tf.io.decode_jpeg(tf.io.read_file(path), channels=3)
        # Ši funkcija palieka reikšmes 0..255; reikiamas mastelis nustatomas modelyje.
        return tf.image.resize(tf.cast(image, tf.float32), [size, size]), label
    ds = ds.map(decode, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.shuffle(len(frame), seed=SEED)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)
train_ds, val_ds, test_ds = image_dataset(train_img, True), image_dataset(val_img), image_dataset(test_img)


In [ ]:
cnn = keras.Sequential([
    keras.Input(shape=(SIZE, SIZE, 3)), layers.Rescaling(1./255),
    layers.Conv2D(32, 3, activation='relu', name='first_conv'), layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu'), layers.MaxPooling2D(2),
    layers.Conv2D(64, 3, activation='relu'), layers.Flatten(),
    layers.Dense(64, activation='relu'), layers.Dropout(.3),
    layers.Dense(len(classes), activation='softmax')], name='flowers_cnn')
cnn.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
cnn.summary()
history_cnn = cnn.fit(train_ds, validation_data=val_ds, epochs=15,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)], verbose=2)
pd.DataFrame(history_cnn.history).to_csv(OUT / 'cnn_mokymo_eiga.csv', index=False)


In [ ]:
def image_metrics(name, model):
    # Testo duomenų tvarka nekeičiama, todėl sutampa su test_img etiketėmis.
    probability = model.predict(test_ds, verbose=0)
    predicted = probability.argmax(axis=1)
    y = test_img.label.to_numpy()
    result = {'modelis': name, 'accuracy': accuracy_score(y, predicted),
              'macro_f1': f1_score(y, predicted, average='macro', zero_division=0)}
    print(classification_report(y, predicted, labels=range(len(classes)), target_names=classes, zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y, predicted, labels=range(len(classes)),
        display_labels=classes, xticks_rotation=35, cmap='Blues')
    plt.title(name + ': nepriklausomas testas'); plt.tight_layout(); plt.show()
    return result

def show_history(history, title):
    frame = pd.DataFrame(history.history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    for ax, metric in zip(axes, ['loss', 'accuracy']):
        ax.plot(frame[metric], label='mokymas')
        ax.plot(frame['val_' + metric], label='validavimas')
        ax.set(xlabel='Epocha (nuo 0)', title=title + ': ' + metric); ax.legend()
    plt.tight_layout(); plt.show()


In [ ]:
show_history(history_cnn, 'CNN')


## 3. Filtrai ir požymių žemėlapiai

Filtras yra išmoktas 3×3×3 svorių rinkinys. Požymių žemėlapis gaunamas slenkant filtrą per vaizdą,
pridedant poslinkį ir taikant ReLU. Žemiau rodomi 16 iš 32 filtrų ir jų aktyvacijos vienai testo nuotraukai.
Filtrai normalizuojami tik vaizdavimui; modelio svoriai nekeičiami.


In [ ]:
sample_images, sample_labels = next(iter(test_ds))
sample = sample_images[:1]
conv = cnn.get_layer('first_conv')
kernels, biases = conv.get_weights()
probe = keras.Model(cnn.inputs, conv.output)
maps = probe(sample, training=False).numpy()
fig, axes = plt.subplots(2, 8, figsize=(13, 4))
for i in range(16):
    kernel = kernels[:, :, :, i]
    axes.flat[i].imshow((kernel - kernel.min()) / (np.ptp(kernel) + 1e-8))
    axes.flat[i].set_title(f'Filtras {i}'); axes.flat[i].axis('off')
plt.tight_layout(); plt.show()
fig, axes = plt.subplots(2, 8, figsize=(13, 4))
for i in range(16):
    axes.flat[i].imshow(maps[0, :, :, i], cmap='viridis')
    axes.flat[i].set_title(f'Žemėlapis {i}'); axes.flat[i].axis('off')
plt.tight_layout(); plt.show()


## 4. Rankinis pirmo sluoksnio skaičiavimas

Kaip originale, vienam filtrui apskaičiuojama 6×6 išvesties sritis.
Keras `Conv2D` iš tiesų atlieka kryžminę koreliaciją (branduolys neapverčiamas).
Vieno taško reikšmė: ReLU(įvesties 3×3×3 lopinėlio ir filtro sandaugų suma + poslinkis).
Svarbu naudoti įvestį po `Rescaling`. `assert_allclose` patikrina sutapimą su tikru sluoksniu.


In [ ]:
scaled = sample.numpy()[0] / 255.0
filter_index = 2
manual = np.zeros((6, 6), dtype='float32')
for row in range(6):
    for column in range(6):
        patch = scaled[row:row+3, column:column+3, :]
        manual[row, column] = max(0.0, float(np.sum(patch * kernels[:, :, :, filter_index]) + biases[filter_index]))
actual = maps[0, :6, :6, filter_index]
np.testing.assert_allclose(manual, actual, atol=1e-4, rtol=1e-4)
print('Rankinė koreliacija sutapo; didžiausias skirtumas:', float(np.max(np.abs(manual - actual))))
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
axes[0].imshow(manual); axes[0].set_title('Rankinis skaičiavimas')
axes[1].imshow(actual); axes[1].set_title('Keras Conv2D')
plt.tight_layout(); plt.show()


## 5. MLP palyginimas

Originalo MLP irgi mokomas tais pačiais vaizdais. Jis iškart išskleidžia visus pikselius,
todėl neturi CNN vietinių filtrų ir svorių bendrinimo. Palyginimui naudojami tie patys
vaizdų dydžiai, padalijimas, maksimali epochų riba ir ankstyvo stabdymo taisyklė.
Parametrų skaičiai skiriasi, todėl tai architektūrų mokomasis, o ne griežtai vienodo biudžeto tyrimas.


In [ ]:
mlp = keras.Sequential([keras.Input(shape=(SIZE, SIZE, 3)), layers.Rescaling(1./255), layers.Flatten(),
    layers.Dense(32, activation='relu'), layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'), layers.Dense(len(classes), activation='softmax')], name='flowers_mlp')
mlp.compile(optimizer=keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
mlp.summary()
history_mlp = mlp.fit(train_ds, validation_data=val_ds, epochs=15,
    callbacks=[keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)], verbose=2)
show_history(history_mlp, 'MLP')
results = pd.DataFrame([image_metrics('CNN', cnn), image_metrics('MLP', mlp)])
results['parametrai'] = [cnn.count_params(), mlp.count_params()]
display(results)
results.to_csv(OUT / 'cnn_mlp_testas.csv', index=False)
cnn.save(OUT / 'flowers_cnn.keras')
reloaded = keras.models.load_model(OUT / 'flowers_cnn.keras')
np.testing.assert_allclose(cnn(sample, training=False).numpy(), reloaded(sample, training=False).numpy(), atol=1e-5)
display(pd.DataFrame(cnn(sample, training=False).numpy(), columns=classes))
winner = results.loc[results.macro_f1.idxmax()]
print(f'Šiame padalijime didžiausias macro F1: {winner["modelis"]}, {winner["macro_f1"]:.3f}.')


## 6. Rezultatų interpretacija

Macro F1 vienodai įvertina kiekvienos klasės F1, o painiavos matrica parodo, kurios gėlės painiojamos.
CNN nėra iš anksto paskelbiamas geresniu: išvada apskaičiuojama pagal tikrą šio paleidimo testą.
Iš vienos imties negalima spręsti apie visas nuotraukų sąlygas. „Teisinga“ softmax suma
nereiškia, kad prognozė teisinga arba tikimybė kalibruota.

Šaltiniai: [TensorFlow CNN](https://www.tensorflow.org/tutorials/images/cnn),
[Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D).
